# RAG Pipeline: WPATH Guidelines + Patient Context

**Project:** Plume Care Navigator

This notebook builds the Retrieval-Augmented Generation (RAG) pipeline that powers the Care Navigator assistant. It:

1. Downloads and chunks the **WPATH Standards of Care v8** PDF
2. Embeds the chunks into a **ChromaDB** vector store using **Voyage AI** embeddings (`voyage-3`)
3. Builds a **LangChain dual retriever** that queries both ChromaDB (guidelines) and BigQuery (`mart_patient_rag_context`)
4. Applies **Presidio PII scrubbing** before any text reaches the LLM (HIPAA guardrail)
5. Runs end-to-end test queries using **Claude 3.5 Haiku** (Anthropic)

### Why Claude + Voyage AI?
- **Claude 3.5 Haiku** is Anthropic's fastest, most cost-efficient model. Its strong instruction-following and safety alignment make it well-suited for clinical decision-support use cases where hallucination risk must be minimised.
- **Voyage AI** (`voyage-3`) is Anthropic's recommended embedding partner, offering state-of-the-art retrieval performance with a generous free tier (200M tokens/month).

### Prerequisites
- Notebook `01_bigquery_ingestion.ipynb` has been run and dbt models are built
- An Anthropic API key (get one free at https://console.anthropic.com)
- A Voyage AI API key (get one free at https://dash.voyageai.com)
- A Google Cloud project with BigQuery access

## 1. Install Dependencies

In [ ]:
%%capture
!pip install langchain langchain-community langchain-anthropic langchain-chroma \
             chromadb voyageai langchain-voyageai pypdf \
             presidio-analyzer presidio-anonymizer \
             google-cloud-bigquery db-dtypes spacy
!python -m spacy download en_core_web_lg --quiet

## 2. Configuration

In [ ]:
import os
from google.colab import userdata

#  Set these in Colab Secrets (key icon in left sidebar) 
# Add ANTHROPIC_API_KEY and VOYAGE_API_KEY to your Colab Secrets
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['VOYAGE_API_KEY']    = userdata.get('VOYAGE_API_KEY')

PROJECT_ID   = 'YOUR_PROJECT_ID'   # Replace with your GCP project ID
MART_DATASET = 'plume_mart'        # Dataset where mart_patient_rag_context lives
CHROMA_DIR   = '/content/vector_db'
WPATH_URL    = 'https://www.wpath.org/media/cms/Documents/SOC%20v8/SOC8_English_final.pdf'

os.makedirs(CHROMA_DIR, exist_ok=True)
print('Configuration set.')

## 3. Google Cloud Authentication

In [ ]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
bq_client = bigquery.Client(project=PROJECT_ID)
print(f'✓ Authenticated — project: {PROJECT_ID}')

## 4. Download & Chunk WPATH SoC v8

In [ ]:
import requests

pdf_path = '/content/wpath_soc_v8.pdf'

if not os.path.exists(pdf_path):
    print(f'Downloading WPATH SoC v8 from:\n  {WPATH_URL}')
    print('  (This may take 1-2 minutes...)')
    r = requests.get(WPATH_URL, timeout=180)
    r.raise_for_status()
    with open(pdf_path, 'wb') as f:
        f.write(r.content)
    print(f'Downloaded {os.path.getsize(pdf_path)/1e6:.1f} MB')
else:
    print(f'PDF already exists at {pdf_path}')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

print('Loading and chunking WPATH SoC v8...')
loader = PyPDFLoader(pdf_path)
pages  = loader.load()
print(f'  Loaded {len(pages)} pages')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' '],
)
chunks = splitter.split_documents(pages)
print(f'Split into {len(chunks)} chunks')
print(f'\nSample chunk (first 300 chars):')
print(chunks[10].page_content[:300])

## 5. Embed Chunks into ChromaDB (Voyage AI)

In [ ]:
from langchain_voyageai import VoyageAIEmbeddings
from langchain_chroma import Chroma

print('Embedding chunks into ChromaDB using Voyage AI voyage-3...')
print(f'Processing {len(chunks)} chunks (free tier: 200M tokens/month)')

# voyage-3 is Anthropic's recommended embedding model.
# It outperforms competing embedding models on MTEB benchmarks
# and has a generous free tier of 200M tokens/month.
embeddings = VoyageAIEmbeddings(
    model='voyage-3',
    voyage_api_key=os.environ['VOYAGE_API_KEY'],
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name='wpath_guidelines',
)

print(f' Persisted {len(chunks)} vectors to {CHROMA_DIR}')

# Quick sanity check
test_results = vectorstore.similarity_search('estradiol dosing for feminising HRT', k=2)
print(f'\nSanity check — top result for "estradiol dosing":')
print(test_results[0].page_content[:300])

## 6. HIPAA Guardrail — Presidio PII Scrubber

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer   = AnalyzerEngine()
anonymizer = AnonymizerEngine()

# Clinical units (e.g. 'pg', 'ng') that Presidio incorrectly tags as LOCATION.
# We build a deny-list to suppress these false positives.
CLINICAL_UNIT_DENY_LIST = {'pg', 'ng', 'mL', 'dL', 'mEq', 'IU', 'mcg', 'mg'}

def scrub_pii(text: str) -> str:
    """
    Scrub PII from text before sending to the LLM.
    Detects and replaces: PERSON, DATE_TIME, LOCATION, EMAIL, PHONE_NUMBER,
    US_SSN, MEDICAL_LICENSE, NRP (national ID).
    Clinical unit abbreviations (pg/mL, ng/dL) are excluded from LOCATION
    detection to prevent false positives corrupting lab values.
    """
    results = analyzer.analyze(
        text=text,
        language='en',
        entities=[
            'PERSON', 'DATE_TIME', 'LOCATION', 'EMAIL_ADDRESS',
            'PHONE_NUMBER', 'US_SSN', 'MEDICAL_LICENSE', 'NRP',
        ],
    )
    # Filter out false positives on clinical unit abbreviations
    filtered = [
        r for r in results
        if text[r.start:r.end].strip() not in CLINICAL_UNIT_DENY_LIST
    ]
    if not filtered:
        return text
    anonymized = anonymizer.anonymize(text=text, analyzer_results=filtered)
    return anonymized.text

# Test the scrubber
test_text = 'Patient Jane Doe, DOB 1995-03-15, SSN 123-45-6789, lives in Austin TX. Estradiol: 148 pg/mL.'
print('Original :', test_text)
print('Scrubbed :', scrub_pii(test_text))
print('\n Note: pg/mL is preserved (clinical unit, not a location)')

## 7. Build the Dual Retriever + LangChain Chain (Claude 3.5 Haiku)

In [ ]:
#ChromaDB retriever 
chroma_retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4},
)

# BigQuery patient context retriever
def get_patient_context(patient_token: str) -> str:
    """
    Fetch the pre-built RAG context text for a patient from the mart.
    The mart_patient_rag_context model already anonymises the data.
    We apply Presidio as an additional guardrail before LLM submission.
    """
    query = f"""
    SELECT rag_context_text
    FROM `{PROJECT_ID}.{MART_DATASET}.mart_patient_rag_context`
    WHERE patient_token = @patient_token
    LIMIT 1
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter('patient_token', 'STRING', patient_token)
        ]
    )
    rows = list(bq_client.query(query, job_config=job_config).result())
    if not rows:
        return 'No patient context found for this token.'
    raw_context = rows[0].rag_context_text
    return scrub_pii(raw_context)   # Apply Presidio guardrail

print('Retrievers configured.')

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# System prompt
SYSTEM_PROMPT = """
You are a clinical decision-support assistant for Plume, a trans-focused telehealth provider.
Your role is to help care coordinators and providers by:
1. Summarising a patient's current clinical state from their patient context
2. Answering clinical questions by citing relevant sections of the WPATH Standards of Care v8

IMPORTANT RULES:
- Always cite the specific WPATH SoC v8 chapter or section you are drawing from
- Never make diagnostic decisions, only surface relevant guideline information
- If the patient context does not contain enough information, say so clearly
- Use respectful, affirming language consistent with trans-inclusive care
- Do not speculate beyond what is in the provided context and guidelines

Patient Context:
{patient_context}

Relevant WPATH SoC v8 Guideline Excerpts:
{guideline_context}
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', '{question}'),
])

# Claude 3.5 Haiku: Anthropic's fastest, most cost-efficient model.
# Ideal for clinical decision-support where speed and instruction-following
# matter more than maximum reasoning depth.
llm = ChatAnthropic(
    model='claude-3-5-haiku-20241022',
    temperature=0.1,   # Low temperature for clinical accuracy
    anthropic_api_key=os.environ['ANTHROPIC_API_KEY'],
    max_tokens=1024,
)

def format_guideline_docs(docs) -> str:
    return '\n\n---\n\n'.join(
        f'[Page {d.metadata.get("page", "?")}] {d.page_content}'
        for d in docs
    )

def build_chain(patient_token: str):
    """Build a LangChain chain for a specific patient."""
    patient_ctx = get_patient_context(patient_token)

    chain = (
        {
            'guideline_context': chroma_retriever | RunnableLambda(format_guideline_docs),
            'patient_context':   RunnableLambda(lambda _: patient_ctx),
            'question':          RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain, patient_ctx

print('LangChain chain builder ready (Claude 3.5 Haiku + Voyage AI).')

## 8. End-to-End Test Queries

In [ ]:
# Test with a synthetic patient token
# In production, this token comes from the Streamlit patient selector.
# For testing, we fetch the first token from the mart.

sample_query = """
SELECT patient_token, rag_context_text
FROM `{project}.{dataset}.mart_patient_rag_context`
WHERE subscription_status = 'Active'
  AND hrt_category = 'Feminising HRT'
LIMIT 1
""".format(project=PROJECT_ID, dataset=MART_DATASET)

rows = list(bq_client.query(sample_query).result())
if rows:
    test_token = rows[0].patient_token
    print(f'Test patient token: {test_token[:16]}...')
    print(f'Context preview: {rows[0].rag_context_text[:200]}...')
else:
    print('No patients found — ensure dbt models have been run.')
    test_token = None

In [ ]:
if test_token:
    chain, patient_ctx = build_chain(test_token)

    print('Patient context (after Presidio scrub):')
    print(patient_ctx)
    print('\n' + '='*60)

    question = 'What does WPATH SoC v8 recommend for monitoring estradiol levels in feminising HRT, and how does this patient compare to those targets?'
    print(f'\nQuestion: {question}')
    print('='*60)

    response = chain.invoke(question)
    print(response)

In [ ]:
if test_token:
    question_2 = 'This patient has not had a visit in 90 days. What does WPATH recommend for follow-up frequency for patients on feminising HRT?'
    print(f'Question: {question_2}')
    print('='*60)
    response_2 = chain.invoke(question_2)
    print(response_2)

## 9. Next Steps

The RAG pipeline is now fully operational. The ChromaDB index is persisted at `/content/vector_db/`.

**To run the full Streamlit UI:**
```bash
cd app/
streamlit run app.py
```

The Streamlit app uses the same `build_chain()` pattern implemented here, with a patient selector sidebar and a live chat interface.

**API Keys needed:**
- `ANTHROPIC_API_KEY` — get free at https://console.anthropic.com
- `VOYAGE_API_KEY` — get free at https://dash.voyageai.com (200M token free tier)